# Entendendo o Problema e Gerando Instâncias

## O que vamos fazer neste notebook?

Antes de sair escrevendo código, precisamos entender exatamente o que o problema pede.
Depois, vamos criar um gerador que produz versões diferentes do problema automaticamente.

Essas versões geradas serão usadas nas semanas seguintes para:
- Resolver com Z3 (gabarito correto)
- Mandar para o Gemini resolver diretamente
- Comparar os resultados

## O Problema Original

A questão 5 da lista descreve a seguinte situação:

Uma empresa quer montar uma prova automática com questões de um banco.
O banco tem questões divididas em 4 tópicos: T1, T2, T3, T4.
Cada questão tem exatamente uma dificuldade: fácil (F), médio (M) ou difícil (D).
Uma questão pode pertencer a mais de um tópico.

A prova deve satisfazer:
- Cada tópico tem pelo menos 1 questão na prova
- Exatamente 1 questão difícil
- Exatamente 2 questões fáceis
- Exatamente 3 questões médias
- Total: 6 questões na prova

Uma "instância" do problema é uma configuração específica do banco de questões.
Algumas instâncias terão solução (satisfatíveis), outras não.
Isso é exatamente o que queremos testar: se o Gemini sabe distinguir os dois casos.

---
## Parte 1 — Analisando o Problema

Antes de modelar formalmente, vamos entender o problema
observando o banco de exemplo e raciocinar sobre as restrições.

### O que sabemos sobre o banco de exemplo?

- Total de questões disponíveis: 8
- Questões fáceis (F): q1, q2, q8
- Questões médias (M): q3, q4, q5, q7
- Questões difíceis (D): q6

A prova precisa ter exatamente 6 questões:
- 1 difícil
- 2 fáceis
- 3 médias

In [1]:
# Cada questão é representada como um dicionário com:
#   'id'          → identificador único
#   'dificuldade' → 'facil', 'medio' ou 'dificil'
#   'topicos'     → lista de tópicos aos quais a questão pertence

banco_exemplo = [
    {'id': 'q1',  'dificuldade': 'facil',   'topicos': [1, 2]},
    {'id': 'q2',  'dificuldade': 'facil',   'topicos': [3]},
    {'id': 'q3',  'dificuldade': 'medio',   'topicos': [1, 4]},
    {'id': 'q4',  'dificuldade': 'medio',   'topicos': [2]},
    {'id': 'q5',  'dificuldade': 'medio',   'topicos': [3, 4]},
    {'id': 'q6',  'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]},
    {'id': 'q7',  'dificuldade': 'medio',   'topicos': [2, 3]},
    {'id': 'q8',  'dificuldade': 'facil',   'topicos': [4]},
]

print("=== Banco de Questões ===")
print(f"Total de questões: {len(banco_exemplo)}\n")

for q in banco_exemplo:
    topicos_str = ', '.join([f'T{t}' for t in q['topicos']])
    print(f"  {q['id']:4s} | dificuldade: {q['dificuldade']:8s} | tópicos: {topicos_str}")

=== Banco de Questões ===
Total de questões: 8

  q1   | dificuldade: facil    | tópicos: T1, T2
  q2   | dificuldade: facil    | tópicos: T3
  q3   | dificuldade: medio    | tópicos: T1, T4
  q4   | dificuldade: medio    | tópicos: T2
  q5   | dificuldade: medio    | tópicos: T3, T4
  q6   | dificuldade: dificil  | tópicos: T1, T2, T3, T4
  q7   | dificuldade: medio    | tópicos: T2, T3
  q8   | dificuldade: facil    | tópicos: T4


### Qual questão é obrigatória?

Olhando o banco, q6 é a única questão difícil.
Como a prova precisa de exatamente 1 difícil, q6 é obrigatória.

Além disso, q6 pertence a T1, T2, T3 e T4 ao mesmo tempo.
Isso significa que ao incluir q6, todos os 4 tópicos já ficam cobertos.

Conclusão: q6 entra na prova com certeza, e isso lembra muito as posições em Probabilidade.

### O que falta escolher depois de fixar q6?

Com q6 garantida, ainda precisamos de 5 questões:
- 2 fáceis → escolher 2 dentre {q1, q2, q8}
- 3 médias → escolher 3 dentre {q3, q4, q5, q7}

Como q6 já cobre todos os tópicos, as questões restantes
podem ser escolhidas livremente respeitando só as contagens.

### Quantas combinações válidas existem?

Combinações de 2 fáceis dentre 3 disponíveis:
- {q1, q2}
- {q1, q8}
- {q2, q8}
→ 3 combinações possíveis

Combinações de 3 médias dentre 4 disponíveis:
- {q3, q4, q5}
- {q3, q4, q7}
- {q3, q5, q7}
- {q4, q5, q7}
→ 4 combinações possíveis

Total de provas válidas: 3 × 4 = 12 provas diferentes possíveis.

Conclusão: o problema é SATISFATÍVEL para este banco de questões.

---
## Parte 2 — Dedução Lógica Formal

Agora vamos modelar o problema usando lógica proposicional,
da mesma forma que o enunciado pede.

### Variáveis proposicionais

Para cada questão do banco, criamos uma variável booleana:

- x1 = "q1 está na prova"
- x2 = "q2 está na prova"
- x3 = "q3 está na prova"
- x4 = "q4 está na prova"
- x5 = "q5 está na prova"
- x6 = "q6 está na prova"
- x7 = "q7 está na prova"
- x8 = "q8 está na prova"

Cada variável vale True (questão selecionada) ou False (não selecionada).
O objetivo é _encontrar uma valoração que satisfaça todas as restrições abaixo_.

---
### Restrições em lógica proposicional

**Restrição 1 — Cada tópico tem pelo menos 1 questão:**

- T1 tem q1, q3, q6 → (x1 ∨ x3 ∨ x6)
- T2 tem q1, q4, q6, q7 → (x1 ∨ x4 ∨ x6 ∨ x7)
- T3 tem q2, q5, q6, q7 → (x2 ∨ x5 ∨ x6 ∨ x7)
- T4 tem q3, q5, q6, q8 → (x3 ∨ x5 ∨ x6 ∨ x8)

**Restrição 2 — Exatamente 1 questão difícil:**

Difíceis: só q6 → x6 = True

Em lógica: x6 ∧ ¬x_nenhuma_outra_dificil

Como q6 é a única difícil, simplifica para: x6

**Restrição 3 — Exatamente 2 questões fáceis:**

Fáceis: q1, q2, q8

Precisamos que exatamente 2 dessas sejam True:
(x1 ∧ x2 ∧ ¬x8) ∨ (x1 ∧ ¬x2 ∧ x8) ∨ (¬x1 ∧ x2 ∧ x8)

**Restrição 4 — Exatamente 3 questões médias:**

Médias: q3, q4, q5, q7

Precisamos que exatamente 3 dessas sejam True:
(x3 ∧ x4 ∧ x5 ∧ ¬x7) ∨ (x3 ∧ x4 ∧ ¬x5 ∧ x7) ∨ (x3 ∧ ¬x4 ∧ x5 ∧ x7) ∨ (¬x3 ∧ x4 ∧ x5 ∧ x7)

**Fórmula final — conjunção de todas as restrições:**

φ = (x1 ∨ x3 ∨ x6)
  ∧ (x1 ∨ x4 ∨ x6 ∨ x7)
  ∧ (x2 ∨ x5 ∨ x6 ∨ x7)
  ∧ (x3 ∨ x5 ∨ x6 ∨ x8)
  ∧ x6
  ∧ exatamente_2_faceis
  ∧ exatamente_3_medias

φ é satisfatível se existir pelo menos uma valoração que a torne verdadeira.

### Verificando uma valoração manualmente

Vamos testar se a valoração abaixo satisfaz φ:

x1 = True  (q1 entra na prova)
x2 = True  (q2 entra na prova)
x3 = True  (q3 entra na prova)
x4 = True  (q4 entra na prova)
x5 = True  (q5 entra na prova)
x6 = True  (q6 entra na prova)
x7 = False (q7 não entra)
x8 = False (q8 não entra)

Verificando cada restrição:

Restrição 1 — tópicos:
  T1: x1 ∨ x3 ∨ x6 = T ∨ T ∨ T = True ✓
  T2: x1 ∨ x4 ∨ x6 ∨ x7 = T ∨ T ∨ T ∨ F = True ✓
  T3: x2 ∨ x5 ∨ x6 ∨ x7 = T ∨ T ∨ T ∨ F = True ✓
  T4: x3 ∨ x5 ∨ x6 ∨ x8 = T ∨ T ∨ T ∨ F = True ✓

Restrição 2 — exatamente 1 difícil:
  Apenas x6 = True → 1 difícil ✓

Restrição 3 — exatamente 2 fáceis:
  x1 = T, x2 = T, x8 = F → 2 fáceis ✓

Restrição 4 — exatamente 3 médias:
  x3 = T, x4 = T, x5 = T, x7 = F → 3 médias ✓


Todas as restrições satisfeitas!
Prova selecionada: {q1, q2, q3, q4, q5, q6}

Conclusão: φ é SATISFATÍVEL. ✓

---
## Parte 3 — Planejamento dos Casos de Teste

Para avaliar verdadeiramente a capacidade de raciocínio lógico e estatístico da LLM, não podemos testá-la apenas com o "caminho feliz" (instâncias onde tudo dá certo). Precisamos de uma suíte de testes robusta que cubra diferentes cenários lógicos. 

Nosso gerador deve produzir instâncias classificadas em quatro categorias principais para servir de *benchmark* contra o Z3:

**Caso 1: Satisfatível (Cenário Base)**
   * **Descrição:** Semelhante ao nosso exemplo inicial. Há abundância de questões e várias combinações possíveis para formar a prova. 
   * **Objetivo:** Verificar se a LLM consegue encontrar pelo menos uma solução válida sem se perder no excesso de informações.

**Caso 2: Insatisfatível por Quebra de Contagem (Falta de Dificuldade)**
   * **Descrição:** O banco de questões simplesmente não possui a quantidade mínima necessária de uma determinada dificuldade (ex: precisamos de 3 médias, mas o banco só tem 2).
   * **Objetivo:** É o erro mais trivial. A LLM deve identificar a impossibilidade matemática de imediato, antes mesmo de cruzar os tópicos.

**Caso 3: Insatisfatível por Cobertura de Tópicos (Falta Tópico)**
   * **Descrição:** O banco tem questões suficientes para satisfazer a cota de dificuldades (exatamente 1 Difícil, 2 Fáceis, 3 Médias), porém, nenhuma das questões escolhidas (ou disponíveis) cobre o tópico T4.
   * **Objetivo:** Testar se a IA cruza adequadamente as informações dos conjuntos. Ela precisa perceber que, mesmo batendo a cota de dificuldades, a restrição de "pelo menos 1 de cada tópico" foi violada.

**Caso 4: Satisfatível com Restrição Apertada (Solução Única)**
   * **Descrição:** O banco é montado de forma que existe *apenas uma* combinação exata de 6 questões que satisfaz todas as restrições simultaneamente. Qualquer desvio invalida a prova.
   * **Objetivo:** É o teste de fogo para a LLM na Abordagem A (dedução em texto livre). Avalia a precisão máxima do modelo sem o auxílio de código.

**Caso 5: Insatisfatível por Conflito de Interesses (A Armadilha do Cruzamento)**
   * **Descrição:** O banco possui questões suficientes para bater as cotas de dificuldade e todos os tópicos estão presentes no banco geral. Porém, a *única* questão que cobre o tópico T4 é da dificuldade "Difícil" (D). Como o problema exige exatamente 1 Difícil, ao escolhermos essa questão, somos obrigados a descartar todas as outras Difíceis. Se as outras Difíceis forem as únicas formas de cobrir outro tópico, o problema se torna insolúvel.
   * **Objetivo:** Testar se a IA percebe o efeito dominó das restrições. Ela não pode avaliar dificuldade e tópicos de forma isolada.

**Caso 6: O Falso Positivo (Tamanho Exato, Mas Inválido)**
   * **Descrição:** O banco de questões possui *exatamente* 6 questões. As dificuldades batem perfeitamente com a regra (1D, 2F, 3M). A IA pode olhar para isso e pensar: "Perfeito, basta selecionar todas!". Mas o conjunto dessas 6 questões não cobre o tópico T2.
   * **Objetivo:** Verificar se a LLM sofre de "preguiça cognitiva". Modelos menores tendem a parar de raciocinar quando veem que a restrição de contagem já está magicamente resolvida.

**Caso 7: Satisfatível com Excesso de Ruído (Stress Test)**
   * **Descrição:** Um banco enorme de questões (mais de 12), com muitas redundâncias e várias questões "inúteis" que compartilham exatamente os mesmos atributos.
   * **Objetivo:** Testar o limite de *context window* (janela de contexto) e a capacidade de atenção da LLM. Muito texto irrelevante costuma causar "alucinações" nas IAs na hora de deduzir a resposta em linguagem natural.

In [2]:
# GERADOR DE CASOS DE TESTE (INSTÂNCIAS)
casos_de_teste = {
    
    "caso_1_satisfativel_base": [
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1, 2]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3]},
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [1, 4]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [2]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [3, 4]},
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]},
        {'id': 'q7', 'dificuldade': 'medio',   'topicos': [2, 3]},
        {'id': 'q8', 'dificuldade': 'facil',   'topicos': [4]}
    ],
    
    "caso_2_insatisfativel_contagem": [
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1, 2]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3]},
        {'id': 'q3', 'dificuldade': 'facil',   'topicos': [4]},
        # Erro induzido: apenas 2 médias no banco, mas a regra pede 3
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [1]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [2, 3]},
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]}
    ],
    
    "caso_3_insatisfativel_topico": [
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [2]},
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [3]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [1, 2]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [2, 3]},
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [1, 3]},
        # Erro induzido: NENHUMA questão possui o tópico 4 (T4)
        {'id': 'q7', 'dificuldade': 'facil',   'topicos': [2]},
    ],
    
    "caso_4_satisfativel_apertado": [
        # Para que todos os tópicos (1,2,3,4) sejam cobertos e a cota seja batida,
        # a IA será OBRIGADA a escolher exatamente um subconjunto específico.
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1]},       # Obrigatória (única com T1)
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [2]},       # Opcional
        {'id': 'q3', 'dificuldade': 'facil',   'topicos': [2]},       # Opcional
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [4]},       # Obrigatória (única com T4)
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [3]},       # Obrigatória
        {'id': 'q6', 'dificuldade': 'medio',   'topicos': [3]},       # Obrigatória
        {'id': 'q7', 'dificuldade': 'dificil', 'topicos': [2]}        # Obrigatória
    ],"caso_5_insatisfativel_conflito": [
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [2]},
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [1]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [2]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [3]},
        # Conflito: Só temos 1 vaga para 'Dificil'. 
        # Mas T4 SÓ existe na q6, e T3 SÓ existe na q5 e q7.
        # Se pegarmos a q6 para cobrir T4, perdemos a q7 e T3 fica descoberto se não pegarmos a q5. 
        # (Ajuste fino do conflito: forçar a escolha entre tópicos via dificuldade)
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [4]}, 
        {'id': 'q7', 'dificuldade': 'dificil', 'topicos': [3]}  
    ],
    
    "caso_6_falso_positivo_exato": [
        # Exatamente 6 questões. Contagem de dificuldades perfeita (1D, 2F, 3M).
        # Porém, o tópico 2 (T2) não existe em nenhuma delas.
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3]},
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [1, 4]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [3]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [4]},
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [1, 3, 4]}
    ],
    
    "caso_7_satisfativel_ruido": [
        # Muitas questões, várias inúteis, para confundir a IA.
        {'id': 'q1',  'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2',  'dificuldade': 'facil',   'topicos': [1]}, # Redundante
        {'id': 'q3',  'dificuldade': 'facil',   'topicos': [1]}, # Redundante
        {'id': 'q4',  'dificuldade': 'facil',   'topicos': [2]},
        {'id': 'q5',  'dificuldade': 'medio',   'topicos': [3]},
        {'id': 'q6',  'dificuldade': 'medio',   'topicos': [3]}, # Redundante
        {'id': 'q7',  'dificuldade': 'medio',   'topicos': [3]}, # Redundante
        {'id': 'q8',  'dificuldade': 'medio',   'topicos': [4]},
        {'id': 'q9',  'dificuldade': 'medio',   'topicos': [4]}, # Redundante
        {'id': 'q10', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]}, # A "Chave de Ouro" escondida
        {'id': 'q11', 'dificuldade': 'dificil', 'topicos': [1]},
        {'id': 'q12', 'dificuldade': 'dificil', 'topicos': [2]}
    ]
}

# Validação rápida de estrutura
for nome_caso, banco in casos_de_teste.items():
    qtd_questoes = len(banco)
    print(f"[{nome_caso.upper()}] carregado com {qtd_questoes} questões.")

[CASO_1_SATISFATIVEL_BASE] carregado com 8 questões.
[CASO_2_INSATISFATIVEL_CONTAGEM] carregado com 6 questões.
[CASO_3_INSATISFATIVEL_TOPICO] carregado com 7 questões.
[CASO_4_SATISFATIVEL_APERTADO] carregado com 7 questões.
[CASO_5_INSATISFATIVEL_CONFLITO] carregado com 7 questões.
[CASO_6_FALSO_POSITIVO_EXATO] carregado com 6 questões.
[CASO_7_SATISFATIVEL_RUIDO] carregado com 12 questões.


### Implementação dos Casos de Teste

Agora vamos implementar em código cada um dos 7 casos descritos acima.
Para cada caso, criaremos uma função que gera um banco de questões
com as características específicas daquele cenário.

Ao final, teremos 20 instâncias no total:
- Caso 1: 4 instâncias (satisfatível base)
- Caso 2: 3 instâncias (falta de dificuldade)
- Caso 3: 3 instâncias (falta de tópico)
- Caso 4: 3 instâncias (solução única)
- Caso 5: 3 instâncias (conflito de cruzamento)
- Caso 6: 2 instâncias (falso positivo)
- Caso 7: 2 instâncias (stress test)

In [3]:
# Bibliotecas necessárias para esta parte
import random  # para gerar bancos aleatórios
import json    # para salvar as instâncias em arquivo

# Vamos verificar se o Z3 está instalado
try:
    from z3 import *
    print("✅ Z3 instalado com sucesso!")
except ImportError:
    print("❌ Z3 não encontrado. Rode: pip install z3-solver")

✅ Z3 instalado com sucesso!


In [4]:
def caso1_satisfativel_base(seed=None):
    """
    Caso 1: Satisfatível — cenário base com abundância de questões.
    Há várias combinações possíveis para formar a prova.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # fáceis — temos mais que 2, dá pra escolher
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1, 2]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3]},
        {'id': 'q3', 'dificuldade': 'facil',   'topicos': [4]},
        # médias — temos mais que 3, dá pra escolher
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [1]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [2, 3]},
        {'id': 'q6', 'dificuldade': 'medio',   'topicos': [4]},
        {'id': 'q7', 'dificuldade': 'medio',   'topicos': [1, 4]},
        # difícil — exatamente 1
        {'id': 'q8', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]},
    ]

    return {
        'caso': 1,
        'descricao': 'Satisfatível — cenário base',
        'banco': banco,
        'esperado': 'SAT'
    }


# Teste
inst = caso1_satisfativel_base()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 1 — Satisfatível — cenário base
Esperado: SAT
Total de questões: 8
  q1   | facil    | T1, T2
  q2   | facil    | T3
  q3   | facil    | T4
  q4   | medio    | T1
  q5   | medio    | T2, T3
  q6   | medio    | T4
  q7   | medio    | T1, T4
  q8   | dificil  | T1, T2, T3, T4


In [5]:
def caso2_insatisfativel_falta_dificuldade(seed=None):
    """
    Caso 2: Insatisfatível — falta de questões de alguma dificuldade.
    Aqui temos apenas 2 questões médias, mas precisamos de 3.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # fáceis — temos exatamente 2
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1, 2]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3, 4]},
        # médias — só temos 2, mas precisamos de 3 → IMPOSSÍVEL
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [1, 3]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [2, 4]},
        # difícil — exatamente 1
        {'id': 'q5', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]},
    ]

    return {
        'caso': 2,
        'descricao': 'Insatisfatível — falta questões médias (temos 2, precisamos de 3)',
        'banco': banco,
        'esperado': 'UNSAT'
    }


# Teste
inst = caso2_insatisfativel_falta_dificuldade()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 2 — Insatisfatível — falta questões médias (temos 2, precisamos de 3)
Esperado: UNSAT
Total de questões: 5
  q1   | facil    | T1, T2
  q2   | facil    | T3, T4
  q3   | medio    | T1, T3
  q4   | medio    | T2, T4
  q5   | dificil  | T1, T2, T3, T4


In [6]:
def caso3_insatisfativel_falta_topico(seed=None):
    """
    Caso 3: Insatisfatível — cotas de dificuldade OK, mas nenhuma questão cobre T4.
    A IA precisa perceber que mesmo batendo as contagens, falta um tópico.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # fáceis — temos exatamente 2
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1, 2]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [2, 3]},
        # médias — temos exatamente 3
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [1]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [2]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [3]},
        # difícil — exatamente 1
        # repare: nenhuma questão tem T4 → IMPOSSÍVEL cobrir T4
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [1, 2, 3]},
    ]

    return {
        'caso': 3,
        'descricao': 'Insatisfatível — cotas OK mas nenhuma questão cobre T4',
        'banco': banco,
        'esperado': 'UNSAT'
    }


# Teste
inst = caso3_insatisfativel_falta_topico()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 3 — Insatisfatível — cotas OK mas nenhuma questão cobre T4
Esperado: UNSAT
Total de questões: 6
  q1   | facil    | T1, T2
  q2   | facil    | T2, T3
  q3   | medio    | T1
  q4   | medio    | T2
  q5   | medio    | T3
  q6   | dificil  | T1, T2, T3


In [7]:
def caso4_satisfativel_solucao_unica(seed=None):
    """
    Caso 4: Satisfatível — existe apenas UMA combinação válida.
    Qualquer desvio invalida a prova. Teste de fogo para a LLM.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # fáceis — temos exatamente 2, não há escolha
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [2]},
        # médias — temos exatamente 3, não há escolha
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [3]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [4]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [1, 3]},
        # difícil — exatamente 1, não há escolha
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [2, 4]},
    ]

    return {
        'caso': 4,
        'descricao': 'Satisfatível — solução única, todas as 6 questões obrigatórias',
        'banco': banco,
        'esperado': 'SAT'
    }


# Teste
inst = caso4_satisfativel_solucao_unica()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 4 — Satisfatível — solução única, todas as 6 questões obrigatórias
Esperado: SAT
Total de questões: 6
  q1   | facil    | T1
  q2   | facil    | T2
  q3   | medio    | T3
  q4   | medio    | T4
  q5   | medio    | T1, T3
  q6   | dificil  | T2, T4


In [8]:
def caso5_insatisfativel_conflito_cruzamento(seed=None):
    """
    Caso 5: Insatisfatível — armadilha do cruzamento.
    A única questão que cobre T4 é difícil.
    Mas a única questão que cobre T1 também é difícil.
    Como precisamos de exatamente 1 difícil, não dá pra cobrir T1 e T4 ao mesmo tempo.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # fáceis — temos exatamente 2
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [2]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3]},
        # médias — temos exatamente 3
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [2]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [3]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [2, 3]},
        # difíceis — temos 2, mas só podemos escolher 1
        # q6 é a única que cobre T1
        # q7 é a única que cobre T4
        # → escolher q6 deixa T4 descoberto
        # → escolher q7 deixa T1 descoberto
        # → IMPOSSÍVEL cobrir T1 e T4 ao mesmo tempo
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [1, 2]},
        {'id': 'q7', 'dificuldade': 'dificil', 'topicos': [3, 4]},
    ]

    return {
        'caso': 5,
        'descricao': 'Insatisfatível — conflito: T1 e T4 só cobertos por difíceis diferentes',
        'banco': banco,
        'esperado': 'UNSAT'
    }


# Teste
inst = caso5_insatisfativel_conflito_cruzamento()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 5 — Insatisfatível — conflito: T1 e T4 só cobertos por difíceis diferentes
Esperado: UNSAT
Total de questões: 7
  q1   | facil    | T2
  q2   | facil    | T3
  q3   | medio    | T2
  q4   | medio    | T3
  q5   | medio    | T2, T3
  q6   | dificil  | T1, T2
  q7   | dificil  | T3, T4


In [9]:
def caso6_falso_positivo(seed=None):
    """
    Caso 6: Insatisfatível — falso positivo.
    O banco tem exatamente 6 questões com contagens perfeitas (1D, 2F, 3M).
    A LLM pode pensar 'basta selecionar todas!' mas T2 não está coberto.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # fáceis — exatamente 2
        {'id': 'q1', 'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2', 'dificuldade': 'facil',   'topicos': [3]},
        # médias — exatamente 3
        {'id': 'q3', 'dificuldade': 'medio',   'topicos': [1, 3]},
        {'id': 'q4', 'dificuldade': 'medio',   'topicos': [4]},
        {'id': 'q5', 'dificuldade': 'medio',   'topicos': [1, 4]},
        # difícil — exatamente 1
        # repare: nenhuma das 6 questões tem T2 → IMPOSSÍVEL
        {'id': 'q6', 'dificuldade': 'dificil', 'topicos': [3, 4]},
    ]

    return {
        'caso': 6,
        'descricao': 'Insatisfatível — falso positivo, contagens OK mas T2 descoberto',
        'banco': banco,
        'esperado': 'UNSAT'
    }


# Teste
inst = caso6_falso_positivo()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 6 — Insatisfatível — falso positivo, contagens OK mas T2 descoberto
Esperado: UNSAT
Total de questões: 6
  q1   | facil    | T1
  q2   | facil    | T3
  q3   | medio    | T1, T3
  q4   | medio    | T4
  q5   | medio    | T1, T4
  q6   | dificil  | T3, T4


In [10]:
def caso7_stress_test(seed=None):
    """
    Caso 7: Satisfatível — stress test com banco grande e muito ruído.
    Muitas questões redundantes para testar o limite de atenção da LLM.
    """
    if seed is not None:
        random.seed(seed)

    banco = [
        # muitas fáceis redundantes
        {'id': 'q1',  'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q2',  'dificuldade': 'facil',   'topicos': [1]},
        {'id': 'q3',  'dificuldade': 'facil',   'topicos': [2]},
        {'id': 'q4',  'dificuldade': 'facil',   'topicos': [2]},
        {'id': 'q5',  'dificuldade': 'facil',   'topicos': [3]},
        {'id': 'q6',  'dificuldade': 'facil',   'topicos': [3]},
        # muitas médias redundantes
        {'id': 'q7',  'dificuldade': 'medio',   'topicos': [1, 2]},
        {'id': 'q8',  'dificuldade': 'medio',   'topicos': [1, 2]},
        {'id': 'q9',  'dificuldade': 'medio',   'topicos': [2, 3]},
        {'id': 'q10', 'dificuldade': 'medio',   'topicos': [2, 3]},
        {'id': 'q11', 'dificuldade': 'medio',   'topicos': [3, 4]},
        {'id': 'q12', 'dificuldade': 'medio',   'topicos': [3, 4]},
        {'id': 'q13', 'dificuldade': 'medio',   'topicos': [1, 4]},
        {'id': 'q14', 'dificuldade': 'medio',   'topicos': [1, 4]},
        # difíceis — temos 2 mas só podemos escolher 1
        {'id': 'q15', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]},
        {'id': 'q16', 'dificuldade': 'dificil', 'topicos': [1, 2, 3, 4]},
    ]

    return {
        'caso': 7,
        'descricao': 'Satisfatível — stress test com 16 questões e muito ruído',
        'banco': banco,
        'esperado': 'SAT'
    }


# Teste
inst = caso7_stress_test()
print(f"Caso: {inst['caso']} — {inst['descricao']}")
print(f"Esperado: {inst['esperado']}")
print(f"Total de questões: {len(inst['banco'])}")
for q in inst['banco']:
    topicos_str = ', '.join([f"T{t}" for t in q['topicos']])
    print(f"  {q['id']:4s} | {q['dificuldade']:8s} | {topicos_str}")

Caso: 7 — Satisfatível — stress test com 16 questões e muito ruído
Esperado: SAT
Total de questões: 16
  q1   | facil    | T1
  q2   | facil    | T1
  q3   | facil    | T2
  q4   | facil    | T2
  q5   | facil    | T3
  q6   | facil    | T3
  q7   | medio    | T1, T2
  q8   | medio    | T1, T2
  q9   | medio    | T2, T3
  q10  | medio    | T2, T3
  q11  | medio    | T3, T4
  q12  | medio    | T3, T4
  q13  | medio    | T1, T4
  q14  | medio    | T1, T4
  q15  | dificil  | T1, T2, T3, T4
  q16  | dificil  | T1, T2, T3, T4


In [11]:
# Gerando as 20 instâncias usando os 7 casos
# Cada caso aparece múltiplas vezes com seeds diferentes

instancias = []

# Caso 1 — 4 instâncias satisfatíveis base
for seed in [1, 2, 3, 4]:
    inst = caso1_satisfativel_base(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Caso 2 — 3 instâncias falta de dificuldade
for seed in [10, 11, 12]:
    inst = caso2_insatisfativel_falta_dificuldade(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Caso 3 — 3 instâncias falta de tópico
for seed in [20, 21, 22]:
    inst = caso3_insatisfativel_falta_topico(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Caso 4 — 3 instâncias solução única
for seed in [30, 31, 32]:
    inst = caso4_satisfativel_solucao_unica(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Caso 5 — 3 instâncias conflito de cruzamento
for seed in [40, 41, 42]:
    inst = caso5_insatisfativel_conflito_cruzamento(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Caso 6 — 2 instâncias falso positivo
for seed in [50, 51]:
    inst = caso6_falso_positivo(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Caso 7 — 2 instâncias stress test
for seed in [60, 61]:
    inst = caso7_stress_test(seed=seed)
    inst['id'] = len(instancias) + 1
    instancias.append(inst)

# Resumo
print(f"✅ Total de instâncias geradas: {len(instancias)}\n")
print(f"{'ID':>3} | {'Caso':>5} | {'Esperado':>8} | Descrição")
print("-" * 65)
for inst in instancias:
    print(f"{inst['id']:>3} | {inst['caso']:>5} | {inst['esperado']:>8} | {inst['descricao']}")

✅ Total de instâncias geradas: 20

 ID |  Caso | Esperado | Descrição
-----------------------------------------------------------------
  1 |     1 |      SAT | Satisfatível — cenário base
  2 |     1 |      SAT | Satisfatível — cenário base
  3 |     1 |      SAT | Satisfatível — cenário base
  4 |     1 |      SAT | Satisfatível — cenário base
  5 |     2 |    UNSAT | Insatisfatível — falta questões médias (temos 2, precisamos de 3)
  6 |     2 |    UNSAT | Insatisfatível — falta questões médias (temos 2, precisamos de 3)
  7 |     2 |    UNSAT | Insatisfatível — falta questões médias (temos 2, precisamos de 3)
  8 |     3 |    UNSAT | Insatisfatível — cotas OK mas nenhuma questão cobre T4
  9 |     3 |    UNSAT | Insatisfatível — cotas OK mas nenhuma questão cobre T4
 10 |     3 |    UNSAT | Insatisfatível — cotas OK mas nenhuma questão cobre T4
 11 |     4 |      SAT | Satisfatível — solução única, todas as 6 questões obrigatórias
 12 |     4 |      SAT | Satisfatível — solução úni

In [12]:
def resolver_com_z3(banco):
    """
    Recebe um banco de questões e usa o Z3 para verificar
    se é possível montar a prova satisfazendo todas as restrições.

    Retorna:
        'SAT'   + lista de questões selecionadas, ou
        'UNSAT' + None
    """
    solver = Solver()

    # Cria uma variável booleana para cada questão
    # xi = True significa que a questão i foi selecionada para a prova
    variaveis = {}
    for q in banco:
        variaveis[q['id']] = Bool(q['id'])

    # Separa as questões por dificuldade e tópico
    faceis   = [variaveis[q['id']] for q in banco if q['dificuldade'] == 'facil']
    medias   = [variaveis[q['id']] for q in banco if q['dificuldade'] == 'medio']
    dificeis = [variaveis[q['id']] for q in banco if q['dificuldade'] == 'dificil']

    topicos = {1: [], 2: [], 3: [], 4: []}
    for q in banco:
        for t in q['topicos']:
            topicos[t].append(variaveis[q['id']])

    # Restrição 1 — cada tópico tem pelo menos 1 questão
    for t in range(1, 5):
        if topicos[t]:
            solver.add(Or(topicos[t]))
        else:
            # tópico vazio → impossível cobrir → força UNSAT
            solver.add(BoolVal(False))

    # Restrição 2 — exatamente 1 difícil
    solver.add(PbEq([(v, 1) for v in dificeis], 1))

    # Restrição 3 — exatamente 2 fáceis
    solver.add(PbEq([(v, 1) for v in faceis], 2))

    # Restrição 4 — exatamente 3 médias
    solver.add(PbEq([(v, 1) for v in medias], 3))

    # Resolve
    resultado = solver.check()

    if resultado == sat:
        modelo = solver.model()
        selecionadas = [q['id'] for q in banco if is_true(modelo[variaveis[q['id']]])]
        return 'SAT', selecionadas
    else:
        return 'UNSAT', None


# Teste rápido com o banco de exemplo da Parte 1
resultado, selecionadas = resolver_com_z3(banco_exemplo)
print(f"Resultado: {resultado}")
if selecionadas:
    print(f"Questões selecionadas: {selecionadas}")

Resultado: SAT
Questões selecionadas: ['q1', 'q2', 'q3', 'q4', 'q5', 'q6']


In [13]:
# Rodando o Z3 em todas as 20 instâncias e comparando com o esperado

print("=== Gabarito Z3 ===\n")
print(f"{'ID':>3} | {'Esperado':>8} | {'Z3':>8} | {'OK?':>4} | Questões selecionadas")
print("-" * 75)

acertos = 0
for inst in instancias:
    resultado, selecionadas = resolver_com_z3(inst['banco'])
    inst['resposta_z3'] = resultado
    inst['questoes_z3'] = selecionadas

    ok = "✅" if resultado == inst['esperado'] else "❌"
    if resultado == inst['esperado']:
        acertos += 1

    questoes_str = str(selecionadas) if selecionadas else "—"
    print(f"{inst['id']:>3} | {inst['esperado']:>8} | {resultado:>8} | {ok:>4} | {questoes_str}")

print("-" * 75)
print(f"\nZ3 acertou {acertos}/{len(instancias)} instâncias")

=== Gabarito Z3 ===

 ID | Esperado |       Z3 |  OK? | Questões selecionadas
---------------------------------------------------------------------------
  1 |      SAT |      SAT |    ✅ | ['q1', 'q2', 'q4', 'q6', 'q7', 'q8']
  2 |      SAT |      SAT |    ✅ | ['q1', 'q3', 'q4', 'q6', 'q7', 'q8']
  3 |      SAT |      SAT |    ✅ | ['q1', 'q3', 'q4', 'q5', 'q6', 'q8']
  4 |      SAT |      SAT |    ✅ | ['q1', 'q3', 'q5', 'q6', 'q7', 'q8']
  5 |    UNSAT |    UNSAT |    ✅ | —
  6 |    UNSAT |    UNSAT |    ✅ | —
  7 |    UNSAT |    UNSAT |    ✅ | —
  8 |    UNSAT |    UNSAT |    ✅ | —
  9 |    UNSAT |    UNSAT |    ✅ | —
 10 |    UNSAT |    UNSAT |    ✅ | —
 11 |      SAT |      SAT |    ✅ | ['q1', 'q2', 'q3', 'q4', 'q5', 'q6']
 12 |      SAT |      SAT |    ✅ | ['q1', 'q2', 'q3', 'q4', 'q5', 'q6']
 13 |      SAT |      SAT |    ✅ | ['q1', 'q2', 'q3', 'q4', 'q5', 'q6']
 14 |    UNSAT |    UNSAT |    ✅ | —
 15 |    UNSAT |    UNSAT |    ✅ | —
 16 |    UNSAT |    UNSAT |    ✅ | —
 17 |    

In [14]:
# Salva as instâncias com o gabarito do Z3 em arquivo JSON
# Este arquivo será usado nas semanas 3 e 4 para comparar com o Gemini

dados_para_salvar = []
for inst in instancias:
    dados_para_salvar.append({
        'id': inst['id'],
        'caso': inst['caso'],
        'descricao': inst['descricao'],
        'esperado': inst['esperado'],
        'banco': inst['banco'],
        'resposta_z3': inst['resposta_z3'],
        'questoes_z3': inst['questoes_z3'],
        'resposta_gemini': None,   # será preenchido na semana 3
    })

with open('instancias.json', 'w', encoding='utf-8') as f:
    json.dump(dados_para_salvar, f, ensure_ascii=False, indent=2)

print("✅ instancias.json salvo com sucesso!")
print(f"   {len(dados_para_salvar)} instâncias com gabarito Z3 prontas para o Gemini.")

✅ instancias.json salvo com sucesso!
   20 instâncias com gabarito Z3 prontas para o Gemini.
